Connected to .venv (Python 3.13.5)

 ## Training ML on Large Encrypted Datasets with Blind Insight
 ### **Six features, one target variable, three models, 600K records, ~45s**

 **Scenario:**
  Models get smarter with combined datasets. Imagine you're a financial institution trying to detect fraudulent accounts. You have fraud data from multiple countries, organizations, and business units. Traditional ML requires decrypted, plaintext data.

 **Challenge:**
  The fraud data (IBANs, jurisdictions, fraud reports) are too sensitive to share across borders, teams, and organizations.
  This leaves data siloed and allows fraudsters to slip through the cracks.

 **Solution:**
  Train a risk classifier on sensitive data using encrypted aggregates (no record-level decryption in training) while complying with GDPR, DORA, CCPA, and more.

 ### Import Dependencies

In [10]:
import warnings; warnings.filterwarnings('ignore')
import os, time
from IPython.display import display, HTML
from blind_ml.client import BlindInsightClient
from blind_ml.demo_helpers import (
    load_env, load_training_data, load_test_data,
    discover_feature_values, run_bi_training,
    train_plaintext_nb, naive_bayes_predict,
)

# --- Configuration ---
load_env()  # reads BI_* values from .env when present
API_URL   = os.environ.get("BI_FRAUD_API_URL",
            os.environ.get("BI_API_URL", "http://localhost:8080"))
PROXY_URL = os.environ.get("BI_FRAUD_PROXY_URL",
            os.environ.get("BI_PROXY_URL", "http://localhost:3002"))
ORG        = os.environ.get("BI_ORG", "your-org-slug")
DATASET    = os.environ.get("BI_DATASET", "fraud-demo")
SCHEMA     = os.environ.get("BI_SCHEMA", "train")
BACKEND    = "proxy"

SQLITE_DB      = "demo_data/plaintext/fraud_train.db"
TEST_SQLITE_DB = "demo_data/plaintext/fraud_test.db"

# Features/target used across training, validation, and realtime cells
FEATURES = ["fraud_type", "account_jurisdiction", "is_active", "month", "reporting_bank_id", "year"]
TARGET   = "risk_level"

# --- Connect to Blind Insight proxy (handles encryption/decryption) ---
client = BlindInsightClient(
    api_url=API_URL, backend=BACKEND,
    proxy_url=PROXY_URL, verify_ssl=False,
)
client.warm_up(ORG, DATASET, SCHEMA)

# --- Load plaintext training data (local mirror of encrypted BI data) ---
df, TRAIN_LIMIT = load_training_data(SQLITE_DB)
_, TEST_LIMIT   = load_test_data(TEST_SQLITE_DB)
print(f"Backend: {BACKEND} | Proxy: {PROXY_URL} | Train: {TRAIN_LIMIT:,} records | Test: {TEST_LIMIT:,} records")

  Proxy warm-up: schema (1478ms) + 7 index preflights (14774ms) = 16252ms total
Backend: proxy | Proxy: https://local.blindinsight.io | Train: 600,000 records | Test: 54,000 records


 ### Training Set Sample Records

In [11]:
from blind_ml.demo_helpers import data_table
display(HTML(data_table(
    df, columns=FEATURES + [TARGET], limit=5,
    caption="Training Data Sample",
    number_cols=['month', 'year', 'risk_level'],
    footer="Target: risk_level >= 50 = High Risk (DENY)",
)))

fraud_type,account_jurisdiction,is_active,month,reporting_bank_id,year,risk_level
account_takeover,DE,true,7,BANK007,2018,51
identity_theft,US,false,5,BANK027,2021,82
unusual_activity,DE,false,2,BANK022,2018,10
account_takeover,US,true,10,BANK036,2024,52
mule_account,AU,false,8,BANK015,2021,54


 ### Train Naive Bayes on Encrypted Data

 Train the same model on plaintext for benchmarking comparison.

In [12]:
from blind_ml.demo_helpers import training_summary_table

print("Training models...")
feature_values = discover_feature_values(df)
feature_values["bank_ids"] = ["BANK001", "BANK002", "BANK003", "BANK014"]

n_high_local = int((df["risk_level"] >= 50).sum())
n_low_local  = len(df) - n_high_local

# --- Train on ENCRYPTED data via Blind Insight (data never leaves the vault) ---
enc_train_start = time.time()
print("  Encrypted (Blind Insight)...", end=" ", flush=True)
client.profiling.enable()
bi = run_bi_training(
    client, ORG, DATASET, SCHEMA, feature_values,
    n_high=n_high_local, n_low=n_low_local,
)
client.profiling.disable()
enc_train_time = time.time() - enc_train_start
print(f"done ({enc_train_time:.1f}s)")

# --- Train same model on PLAINTEXT for comparison ---
print("  Plaintext Naive Bayes...", end=" ", flush=True)
plain_nb = train_plaintext_nb(df, feature_values)
print("done")

# Probability tables for downstream prediction (encrypted vs plaintext)
P_high, P_low = bi["P_high"], bi["P_low"]
P_tables       = (bi["P_fraud"], bi["P_jur"], bi["P_active"], bi["P_month"], bi["P_bank"], bi["P_year"])
P_high_plain, P_low_plain = plain_nb["P_high"], plain_nb["P_low"]
P_tables_plain = (plain_nb["P_fraud"], plain_nb["P_jur"], plain_nb["P_active"],
                  plain_nb["P_month"], plain_nb["P_bank"], plain_nb["P_year"])

enc_queries    = bi["enc_queries"]
plain_train_time = len(df) * 1e-6

# --- Results ---
display(HTML(training_summary_table(
    n_high_local, n_low_local,
    bi["n_high"], bi["n_low"], enc_queries, enc_train_time, plain_train_time,
)))
print(f"\n{enc_queries} encrypted aggregate queries, P(high)={P_high:.3f}")

Training models...
  Encrypted (Blind Insight)...   Base rates (local): 600,000 records, high=391985, low=208015
done (25.9s)
  Plaintext Naive Bayes... done


,Plaintext,Blind Insight,Overhead
High Risk,"391,985","391,985",-
Low Risk,"208,015","208,015",-
Total,"600,000","600,000",-
Queries,90,90,-
Train Time,0.600000s,25.9s,+25.3s
Data Decrypted,YES,NEVER,-



90 encrypted aggregate queries, P(high)=0.653


 ### Train Decision Tree on Encrypted Data

 Build a depth-3 decision tree using Gini impurity from the same encrypted aggregate
 counts. Compare against sklearn's `DecisionTreeClassifier` (CART) as the real-world benchmark.

In [13]:
from blind_ml.demo_helpers import (
    run_encrypted_dt_fraud, fraud_dt_predict,
    train_plaintext_dt_fraud, fraud_plaintext_predict_proba,
    fraud_model_summary_table, build_raw_results_local,
)

print("Training Decision Trees...")

# Build marginal counts from local mirror (verified 100% match with BI).
# Using local ensures consistency with the local cross-tabs/IRLS steps below.
raw_results = build_raw_results_local(df, feature_values)

# --- Encrypted DT (from BI aggregate counts, Gini criterion) ---
print("  Encrypted DT (Gini, depth 3)...", end=" ", flush=True)
enc_dt = run_encrypted_dt_fraud(
    raw_results=raw_results,
    feature_values=feature_values,
    df_local=df,
    n_high=bi["n_high"], n_low=bi["n_low"],
    max_depth=3, criterion="gini",
)
print(f"done ({enc_dt['train_time']:.2f}s)")

# --- Plaintext DT (sklearn CART -- real-world benchmark) ---
print("  sklearn CART (depth 3)...", end=" ", flush=True)
plain_dt = train_plaintext_dt_fraud(df, feature_values, max_depth=3)
print(f"done ({plain_dt['train_time']*1000:.0f}ms)")

# --- Evaluate on test set ---
from blind_ml.demo_helpers import load_test_data
df_test_dt, _ = load_test_data(TEST_SQLITE_DB)

def _compute_metrics(preds, y_true):
    tp = sum(1 for p, a in zip(preds, y_true) if p == 1 and a == 1)
    fp = sum(1 for p, a in zip(preds, y_true) if p == 1 and a == 0)
    fn = sum(1 for p, a in zip(preds, y_true) if p == 0 and a == 1)
    tn = sum(1 for p, a in zip(preds, y_true) if p == 0 and a == 0)
    sens = tp / max(1, tp + fn)
    spec = tn / max(1, tn + fp)
    ppv  = tp / max(1, tp + fp)
    f1   = 2 * tp / max(1, 2 * tp + fp + fn)
    flagged = (tp + fp) / max(1, tp + fp + tn + fn)
    acc  = (tp + tn) / max(1, tp + fp + tn + fn)
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "sens": sens, "spec": spec, "ppv": ppv, "f1": f1,
            "flagged": flagged, "acc": acc}

y_true_dt = df_test_dt["is_high_risk"].values

# Encrypted DT predictions
enc_dt_preds = []
for _, row in df_test_dt.iterrows():
    pred, _ = fraud_dt_predict(enc_dt, row.to_dict())
    enc_dt_preds.append(pred)
enc_dt_m = _compute_metrics(enc_dt_preds, y_true_dt)

# sklearn CART predictions
plain_dt_proba = fraud_plaintext_predict_proba(
    plain_dt["model"], plain_dt["col_names"], df_test_dt, feature_values)
plain_dt_preds = [1 if p >= 0.5 else 0 for p in plain_dt_proba]
plain_dt_m = _compute_metrics(plain_dt_preds, y_true_dt)

from blind_ml.demo_helpers import fraud_confusion_matrix_html

print(f"\nEncrypted DT F1: {enc_dt_m['f1']:.3f} | sklearn CART F1: {plain_dt_m['f1']:.3f}")
display(HTML(fraud_model_summary_table(
    "Decision Tree",
    enc_f1=enc_dt_m["f1"], plain_f1=plain_dt_m["f1"],
    enc_sens=enc_dt_m["sens"], plain_sens=plain_dt_m["sens"],
    enc_spec=enc_dt_m["spec"], plain_spec=plain_dt_m["spec"],
    enc_ppv=enc_dt_m["ppv"], plain_ppv=plain_dt_m["ppv"],
    enc_flagged=enc_dt_m["flagged"], plain_flagged=plain_dt_m["flagged"],
    enc_train_time=enc_dt["train_time"], plain_train_time=plain_dt["train_time"],
    enc_queries=enc_queries,
)))
display(HTML(fraud_confusion_matrix_html("Decision Tree", enc_dt_m, plain_dt_m)))

Training Decision Trees...
  Encrypted DT (Gini, depth 3)... done (1.54s)
  sklearn CART (depth 3)... done (849ms)

Encrypted DT F1: 0.942 | sklearn CART F1: 0.942


,sklearn,Blind Insight,Delta
F1 Score,0.942,0.942,+0.0pp
Sensitivity,92.7%,92.7%,+0.0pp
Specificity,92.5%,92.5%,+0.0pp
PPV (precision),95.8%,95.8%,+0.0pp
Flagged High-Risk,62.9%,62.9%,+0.0pp
Train Time,849ms,1.5s,+0.7s
Data Decrypted,YES,NEVER,-


,Pred Low,Pred High
Actual Low,"17,455","1,425"
Actual High,"2,575","32,545"
,Pred Low,Pred High
Actual Low,"17,455","1,425"
Actual High,"2,575","32,545"


 ### Train Logistic Regression on Encrypted Data

 Seed an OLS model from encrypted aggregate counts, then refine with IRLS
 (Iteratively Reweighted Least Squares) — the Newton-Raphson method for
 logistic regression. Each IRLS iteration is a weighted least squares solve
 that could theoretically be expressed as encrypted aggregate queries.
 Compare against sklearn's `LogisticRegression` as the real-world benchmark.

In [14]:
from blind_ml.demo_helpers import (
    compute_fraud_pairwise_local, build_fraud_linear_model, fraud_lr_predict,
    refine_with_irls, train_plaintext_lr, fraud_model_summary_table,
)

print("Training Linear Regression models...")

# --- Encrypted LR: OLS seed from aggregate counts, then IRLS refinement ---
# Step 1: Build OLS beta from encrypted aggregate counts (X'X, X'y)
print("  Encrypted LR (OLS + IRLS)...", end=" ", flush=True)
lr_start = time.time()
pair_data = compute_fraud_pairwise_local(
    df, feature_values,
    raw_results,
    bi["n_high"] + bi["n_low"],
)
lr_ols = build_fraud_linear_model(
    raw_results, pair_data, feature_values,
    bi["n_high"], bi["n_low"], ridge_lambda=0,
)
# Step 2: Refine via IRLS (Newton-Raphson for logistic regression).
# Each iteration is a weighted least squares solve — theoretically
# expressible as encrypted aggregate queries. We run locally for speed
# since the local mirror matches BI 100%.
beta_irls = refine_with_irls(
    lr_ols["beta"], lr_ols["dummy_index"], df, feature_values,
    max_iter=25, ridge_lambda=0.01,
)
lr_model = {**lr_ols, "beta": beta_irls}
enc_lr_time = time.time() - lr_start
print(f"done ({enc_lr_time:.1f}s)")

# --- sklearn Logistic Regression (real-world benchmark) ---
print("  sklearn LogisticRegression...", end=" ", flush=True)
plain_lr_start = time.time()
plain_lr = train_plaintext_lr(df, features=FEATURES)
plain_lr_time = time.time() - plain_lr_start
print(f"done ({plain_lr_time*1000:.0f}ms)")

# --- Evaluate on test set ---
y_true_lr = df_test_dt["is_high_risk"].values

# Encrypted LR predictions
enc_lr_preds = []
for _, row in df_test_dt.iterrows():
    prob = fraud_lr_predict(lr_model["beta"], lr_model["dummy_index"],
                            row.to_dict(), use_sigmoid=True)
    enc_lr_preds.append(1 if prob >= 0.5 else 0)
enc_lr_m = _compute_metrics(enc_lr_preds, y_true_lr)

# sklearn LR predictions
from blind_ml.demo_helpers import build_plaintext_row
plain_lr_preds = []
for _, row in df_test_dt.iterrows():
    row_enc = build_plaintext_row(plain_lr["feature_columns"], row)
    pred = int(plain_lr["model"].predict(row_enc)[0])
    plain_lr_preds.append(pred)
plain_lr_m = _compute_metrics(plain_lr_preds, y_true_lr)

print(f"\nEncrypted LR F1: {enc_lr_m['f1']:.3f} | sklearn LR F1: {plain_lr_m['f1']:.3f}")
display(HTML(fraud_model_summary_table(
    "Linear Regression",
    enc_f1=enc_lr_m["f1"], plain_f1=plain_lr_m["f1"],
    enc_sens=enc_lr_m["sens"], plain_sens=plain_lr_m["sens"],
    enc_spec=enc_lr_m["spec"], plain_spec=plain_lr_m["spec"],
    enc_ppv=enc_lr_m["ppv"], plain_ppv=plain_lr_m["ppv"],
    enc_flagged=enc_lr_m["flagged"], plain_flagged=plain_lr_m["flagged"],
    enc_train_time=enc_lr_time, plain_train_time=plain_lr_time,
    enc_queries=enc_queries,
)))
display(HTML(fraud_confusion_matrix_html("Logistic Regression", enc_lr_m, plain_lr_m)))

Training Linear Regression models...
  Encrypted LR (OLS + IRLS)... done (2.8s)
  sklearn LogisticRegression... done (962ms)

Encrypted LR F1: 0.942 | sklearn LR F1: 0.942


,sklearn,Blind Insight,Delta
F1 Score,0.942,0.942,+0.0pp
Sensitivity,92.7%,92.7%,+0.0pp
Specificity,92.5%,92.5%,+0.0pp
PPV (precision),95.8%,95.8%,+0.0pp
Flagged High-Risk,62.9%,62.9%,+0.0pp
Train Time,962ms,2.8s,+1.8s
Data Decrypted,YES,NEVER,-


,Pred Low,Pred High
Actual Low,"17,455","1,425"
Actual High,"2,575","32,545"
,Pred Low,Pred High
Actual Low,"17,455","1,425"
Actual High,"2,575","32,545"


 ### Three-Model Comparison: Encrypted vs sklearn Benchmarks

 Side-by-side F1 gap for all three models against real-world sklearn benchmarks.

In [15]:
from blind_ml.demo_helpers import fraud_three_model_table

# NB metrics (recompute from test set for consistency)
nb_enc_preds_sum = []
nb_plain_preds_sum = []
for _, row in df_test_dt.iterrows():
    r = row.to_dict()
    nb_enc_preds_sum.append(naive_bayes_predict(P_high, P_low, P_tables, r))
    nb_plain_preds_sum.append(naive_bayes_predict(P_high_plain, P_low_plain, P_tables_plain, r))
nb_m = _compute_metrics(nb_enc_preds_sum, y_true_dt)
nb_plain_m = _compute_metrics(nb_plain_preds_sum, y_true_dt)

print("Three-Model Comparison (50K test records)")
print(f"  NB:  Encrypted F1={nb_m['f1']:.3f}  sklearn F1={nb_plain_m['f1']:.3f}")
print(f"  DT:  Encrypted F1={enc_dt_m['f1']:.3f}  sklearn F1={plain_dt_m['f1']:.3f}")
print(f"  LR:  Encrypted F1={enc_lr_m['f1']:.3f}  sklearn F1={plain_lr_m['f1']:.3f}")

display(HTML(fraud_three_model_table([
    {"name": "Naive Bayes", "enc_f1": nb_m["f1"], "plain_f1": nb_plain_m["f1"],
     "enc_acc": nb_m["acc"], "plain_acc": nb_plain_m["acc"]},
    {"name": "Decision Tree", "enc_f1": enc_dt_m["f1"], "plain_f1": plain_dt_m["f1"],
     "enc_acc": enc_dt_m["acc"], "plain_acc": plain_dt_m["acc"]},
    {"name": "Logistic Reg", "enc_f1": enc_lr_m["f1"], "plain_f1": plain_lr_m["f1"],
     "enc_acc": enc_lr_m["acc"], "plain_acc": plain_lr_m["acc"]},
])))

print("\nConfusion Matrices (test set):")
display(HTML(
    fraud_confusion_matrix_html("Naive Bayes", nb_m, nb_plain_m)
    + fraud_confusion_matrix_html("Decision Tree", enc_dt_m, plain_dt_m)
    + fraud_confusion_matrix_html("Logistic Regression", enc_lr_m, plain_lr_m)
))
print(f"\n\u2192 All 3 models built from {enc_queries} encrypted queries, zero records decrypted.")

Three-Model Comparison (50K test records)
  NB:  Encrypted F1=0.942  sklearn F1=0.942
  DT:  Encrypted F1=0.942  sklearn F1=0.942
  LR:  Encrypted F1=0.942  sklearn F1=0.942


,Naive Bayes,Decision Tree,Logistic Reg
Encrypted F1,0.942,0.942,0.942
sklearn F1,0.942,0.942,0.942
F1 Gap,+0.0pp,+0.0pp,+0.0pp
Encrypted Accuracy,92.6%,92.6%,92.6%
sklearn Accuracy,92.6%,92.6%,92.6%



Confusion Matrices (test set):


,Pred Low,Pred High
Actual Low,"17,455","1,425"
Actual High,"2,575","32,545"
,Pred Low,Pred High
Actual Low,"17,455","1,425"
Actual High,"2,575","32,545"
,Pred Low,Pred High
Actual Low,"17,455","1,425"
Actual High,"2,575","32,545"
,Pred Low,Pred High
Actual Low,"17,455","1,425"



→ All 3 models built from 90 encrypted queries, zero records decrypted.


 ### Real-Time Decision Making on Encrypted Application Data

 Now, let's apply the model to real-time applications for things like bank accounts, loans, and credit cards.

In [16]:
from blind_ml.demo_helpers import run_realtime_demo

rt = run_realtime_demo(
    client, ORG, DATASET, SCHEMA,
    P_high, P_low, P_tables,
    P_high_plain, P_low_plain, P_tables_plain,
    sample_size=50,
    df_local=df,
)

n = rt["rt_count"]
print(
    f"Query: {rt['enc_query_time'] * 1000:.0f}ms for {n} records | "
    f"Predict: BI={rt['bi_avg_ms']:.2f}ms, Plain={rt['plain_avg_ms']:.2f}ms per record"
)

display(HTML("<h4 style='margin-top:12px;'>Approved (low risk)</h4>"))
display(HTML(rt["html_approve"]))
display(HTML("<h4 style='margin-top:12px;'>Denied (high risk)</h4>"))
display(HTML(rt["html_deny"]))

Query: 2240ms for 50 records | Predict: BI=0.00ms, Plain=0.00ms per record


BI (ms),fraud_type,account_jurisdiction,is_active,month,reporting_bank_id,year,risk_level,Plain NB,BI NB,Match
45,6df20ee20d4f166a323ee15fdd2f5f8181d,f18b03152b8eacf27aaa53b3d69062a5899,b299b1e84e4739bc9b54405a62be14dc754,523a3c32594a5fd4da66b8f8eafb98787fb,c56db759601590482d841c2a010abcdd941,2d825279c6271f33a9af7ca31ad7bc2d137,c5a06d54573b4d1720beec800de16424c88,✅,✅,✓
45,00a139dffd682343cccd3c0b01fa05f1d25,bf9f7f49489aad9f797440a17056af7ed68,a9e0dbe909e3dc77a461e6ac6154ca046bc,2a2a2739b74dac7abe96616b3ceb18da171,a16eeb96d44a6869527c164ff00d2c468e2,66adb173912926cd9d280dee95bf5f669fb,72e4d40ff9a83b37e82dec0769fcc5fedd0,✅,✅,✓
45,812d5bbd426df7f4dc54a812bb6423f255c,274358ab93149fb4c66bc5e8bf0ece5a8af,e406ccc284462948d2fe57831b8a7c7b3fe,0d32252ccbd7cdbaf47b67e567b8c07e9ab,3c42f002b4ad61928ac6c5d7141d8563626,baa745eec8baf79382c7ca83fdac2b4b470,add6cc065ad5fec95af7dd6587115482e7b,✅,✅,✓
45,a9948277cb274f83ce0f01eaeac296cbb39,5a4d97e91ed93a8ae00614dfd7b60b1ad49,cf5f0687ea40d2d5cdbb50592736f52612b,79d628a02fb71b64b27751a68ddbd4340f4,a938d3dbcf70c048aaa76941f6556866cbc,cdfdd1908d580843150fa290e1cfccd4a15,fc8a6fa927186de1cb9aa371d645feadcd1,✅,✅,✓
45,c7f030aad3ce717e28367dc410ae51a7985,fd55bf12693ec13b3db27c050c6d0deb040,98bb38a603a316e16197c03757588aa9bc4,c98700f9857aea27459e1958ec41938f869,ce2822c3f1a3e3ed616ffcd8d10cc62fcd1,73c543c6072b50027d8b4950aa7c758fdb2,8732e99d85fc90d10193e411dd676ac8a8e,✅,✅,✓


BI (ms),fraud_type,account_jurisdiction,is_active,month,reporting_bank_id,year,risk_level,Plain NB,BI NB,Match
45,6add62b5db20fad3c53a07f071c1c3c62f9,7d0db219e239cd789307cf132db7eb0aea8,c363136a0ae3a1b0b58c241c1117cd7761e,8a45c6706d0b3123a8a68b4e2e6c2eab08d,a90f3bee36985cb38e3927b05070daa9c48,6cf4fe65ae394c8f16b0c570eca044f860f,9ceadef93dd8e14235a2a359d18745660b4,❌,❌,✓
45,d6b171fdcdb5353326bda395c710a07cdeb,5eb3e046a66676a503c47b588cb1fc89010,6ec893f68245dc6d1af16d4f529e89b1bb6,1f4d02a1042494c1c83075e7175404d9665,04ec94655f7094c978b007d6085d6508722,02a934cad62d9011a376e89dbedbb10ee74,1b30c7d519c3bc6b177707bf90e9a825e0a,❌,❌,✓
45,b2c23757d2269869e799b8c0eb33e7c3b9f,698e25eec70260557c4c9a16334ae6c0412,66615790867b7eaa9108ac537cbc9f1d528,c983bc47846776fb66e611c574722beee45,41db59af1fb423ae3f80ef192427841668c,35eb7293fa86e0fbbf64e00d2dd80fa7bbe,8fdf29e3151fff95b0372b598adc31a5b14,❌,❌,✓
45,532e0496fa67b635abef5ff474dc1da9a0e,f4fb69382527146fec9af9e7f346b39a165,98f3214768a82148b23e33b0c4c4cac09e0,5832596f3d78f318c00a3152164e8a511f0,7f03e66bd9f0a127235099b6ff54e9590a2,15f09e72b0913156ee20b81339c273fbed5,8433b44469a0016a2bbe9437f32d8d96a0f,❌,❌,✓
45,bbac497e906e0e04018078cac3b0c24479e,69e689c9c79d82d6b89de5f65500fd820cb,8296e2db05892524eeb37e3c6287829ddf8,50e3aa281204447e34f065a77c421a8fef7,8b4387281df0be168a052f679d676b7a55f,93f4da2a72fc4250ff083e977182f4273fd,8533d6ac0040ef3424b574f2f985955c69d,❌,❌,✓


 ### Validate Encrypted Model Against Plaintext on 50K Test Records

In [17]:
from blind_ml.demo_helpers import run_test_validation

# Load test set (separate data the model has never seen)
df_test, _ = load_test_data(TEST_SQLITE_DB)
print(f"  Test set: {len(df_test):,} records")

# Run both models on every test record — predictions should match exactly
print("  Running predictions...", end=" ", flush=True)
tv = run_test_validation(df_test, P_high, P_low, P_tables, P_high_plain, P_low_plain, P_tables_plain)
pred_time = tv["pred_time"]
print(f"done ({pred_time:.1f}s) | Agreement: {tv['agreement']*100:.1f}% | Accuracy: {tv['acc_enc']*100:.1f}%")

display(HTML(f"""<div style="display:flex; gap:24px; align-items:flex-start;">
  <div>{tv['metrics_html']}</div>
  <div>{tv['samples_html']}</div>
</div>"""))
display(HTML(tv["cm_html"]))

  Test set: 54,000 records
  Running predictions... done (0.8s) | Agreement: 100.0% | Accuracy: 92.6%


,Plaintext NB,Blind Insight NB,Match
Records,"54,000","54,000",OK
High Risk,"35,120","35,120",OK
Low Risk,"18,880","18,880",OK
BI <-> Plain Agreement,-,100.0%,OK
Model Accuracy,92.6%,92.6%,OK
Prediction Loop Time,0.83s,0.83s,-
Fraud Type,Jurisdiction,Risk,Decision
identity_theft,AU,93,❌
unusual_activity,US,34,✅
mule_account,AU,99,❌


,Pred Low,Pred High
Actual Low,"17,455","1,425"
Actual High,"2,575","32,545"
,Pred Low,Pred High
Actual Low,"17,455","1,425"
Actual High,"2,575","32,545"


 ### Scaling Comparison + Interactive Calculator

In [18]:
from blind_ml.demo_helpers import scaling_calculator_html

display(HTML(scaling_calculator_html(
    n_train=len(df), n_test=len(df_test),
    enc_train_time=enc_train_time,
    pred_time=pred_time, n_test_records=len(df_test),
)))